In [1]:
import pandas as pd
import os
import functions_stanza
import multiprocessing

/home/christopher/projects/corpus_tryout/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-10 16:10:36 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-10 16:10:36 INFO: Downloaded file to /home/christopher/stanza_resources/resources.json
2026-03-10 16:10:38 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | com

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
dfs = []

for file in os.listdir("ingforms_download"):
    csv = pd.read_csv(f"ingforms_download/{file}", sep=";", header=None)
    csv = csv.set_index(csv[0])
    csv = csv.drop(columns=[0])
    dfs.append(csv)

In [4]:
df = pd.concat(dfs)

In [5]:
df = df.sort_index()

In [6]:
df = df.drop_duplicates().reset_index(drop=True) # remove duplicates; export from kontext pretty iffy

In [7]:
df = df.fillna("")

In [8]:
df["english"] = df[2] + " " + df[3] + " " + df[4]

In [ ]:
replacements = [
    ("‘ ", "‘"),
    (" ’", "’"),
    (" ,", ","),
    (" .", "."),
    (" :", ":"),
    (" ?", "?"),
    ("( ", "("),
    (" )", ")"),
    (" ;", ";"),
    ("' ", "'"),
    (" '", "'"),
    (" -", "-"),
    ("- ", "-"),
    ("ş", "ș"),
    ("ţ", "ț"),  # note: the last two have not been considered in current pickle data!
]

In [10]:
for replacement in replacements:
    df["english"] = df["english"].str.replace(replacement[0], replacement[1])

In [11]:
for replacement in replacements:
    df[7] = df[7].str.replace(replacement[0], replacement[1])

In [12]:
CHUNK_SIZE = 2500

# Parse English

In [14]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_en = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks, args=(df, "en", "english", "ingforms", CHUNK_SIZE, 9)
)

In [16]:
process_en.start()

2026-03-05 12:37:58 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-05 12:37:58 INFO: Downloaded file to /home/christopher/stanza_resources/resources.json
2026-03-05 12:37:59 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-03-05 12:37:59 INFO: Using device: cpu
2026-03-05 12:37:59 INFO: Loading: tokenize
2026-03-05 12:38:00 INFO: Loading: mwt
2026-03-05 12:38:00 INFO: Loading: pos

Parsing chunk 20000 to 22499


In [17]:
multiprocessing.active_children()

[<Process name='Process-1' pid=257950 parent=256737 started>]

In [13]:
parsed_dict_en = functions_stanza.load_pickled_data(folder="ingforms", language="en")

parsed_slice_en_27500_29414.pkl
parsed_slice_en_17500_19999.pkl
parsed_slice_ro_20000_22499.pkl
parsed_slice_en_7500_9999.pkl
parsed_slice_ro_17500_19999.pkl
parsed_slice_ro_10000_12499.pkl
parsed_slice_en_20000_22499.pkl
parsed_slice_ro_0_2499.pkl
parsed_slice_ro_15000_17499.pkl
parsed_slice_en_12500_14999.pkl
parsed_slice_en_2500_4999.pkl
parsed_slice_ro_5000_7499.pkl
parsed_slice_ro_25000_27499.pkl
parsed_slice_en_10000_12499.pkl
parsed_slice_en_25000_27499.pkl
parsed_slice_ro_22500_24999.pkl
parsed_slice_ro_12500_14999.pkl
parsed_slice_en_0_2499.pkl
parsed_slice_ro_27500_29414.pkl
parsed_slice_ro_7500_9999.pkl
parsed_slice_en_22500_24999.pkl
parsed_slice_ro_2500_4999.pkl
parsed_slice_en_15000_17499.pkl
parsed_slice_en_5000_7499.pkl


In [14]:
adv_part_indices_and_lemmas = []

In [15]:
number = 1

for i in range(0, len(df)):
    row = df.iloc[i]
    contains_adv_part, lemma_of_adv_part = functions_stanza.sentences_contain_adv_part(
        row[3], parsed_dict_en[i]
    )
    if contains_adv_part:
        adv_part_indices_and_lemmas.append((i, lemma_of_adv_part))

{
  "id": 17,
  "text": "hurrying",
  "lemma": "hurry",
  "upos": "VERB",
  "xpos": "VBG",
  "feats": "Tense=Pres|VerbForm=Part",
  "head": 15,
  "deprel": "advcl",
  "start_char": 6962,
  "end_char": 6970
}
{
  "id": 28,
  "text": "wondering",
  "lemma": "wonder",
  "upos": "VERB",
  "xpos": "VBG",
  "feats": "Tense=Pres|VerbForm=Part",
  "head": 22,
  "deprel": "advcl",
  "start_char": 7754,
  "end_char": 7763
}
{
  "id": 21,
  "text": "finding",
  "lemma": "find",
  "upos": "VERB",
  "xpos": "VBG",
  "feats": "Tense=Pres|VerbForm=Part",
  "head": 63,
  "deprel": "advcl",
  "start_char": 9418,
  "end_char": 9425
}
{
  "id": 5,
  "text": "finding",
  "lemma": "find",
  "upos": "VERB",
  "xpos": "VBG",
  "feats": "Tense=Pres|VerbForm=Part",
  "head": 12,
  "deprel": "advcl",
  "start_char": 10220,
  "end_char": 10227
}
{
  "id": 9,
  "text": "sending",
  "lemma": "send",
  "upos": "VERB",
  "xpos": "VBG",
  "feats": "Tense=Pres|VerbForm=Part",
  "head": 7,
  "deprel": "advcl",
  "start

In [16]:
adv_part_indices_and_lemmas

[(27, 'hurry'),
 (31, 'wonder'),
 (36, 'find'),
 (41, 'find'),
 (56, 'send'),
 (58, 'lie'),
 (65, 'mutter'),
 (68, 'talk'),
 (83, 'try'),
 (89, 'lick'),
 (105, 'turn'),
 (106, 'rise'),
 (112, 'point'),
 (113, 'call'),
 (114, 'turn'),
 (115, 'say'),
 (116, 'look'),
 (117, 'turn'),
 (119, 'look'),
 (124, 'get'),
 (128, 'address'),
 (131, 'remark'),
 (134, 'hope'),
 (136, 'trot'),
 (152, 'say'),
 (159, 'take'),
 (162, 'forget'),
 (171, 'scratch'),
 (173, 'say'),
 (184, 'try'),
 (186, 'know'),
 (187, 'think'),
 (190, 'run'),
 (198, 'smoke'),
 (200, 'make'),
 (201, 'swallow'),
 (207, 'rear'),
 (209, 'remark'),
 (211, 'try'),
 (222, 'hatch'),
 (224, 'raise'),
 (234, 'nibble'),
 (244, 'judge'),
 (246, 'say'),
 (247, 'change'),
 (250, 'stare'),
 (262, 'nurse'),
 (264, 'stir'),
 (270, 'feel'),
 (275, 'jump'),
 (284, 'fling'),
 (299, 'wave'),
 (300, 'wave'),
 (306, 'expect'),
 (308, 'sit'),
 (311, 'begin'),
 (318, 'rest'),
 (326, 'turn'),
 (328, 'shake'),
 (330, 'look'),
 (334, 'turn'),
 (337, '

# Parse Romanian

In [ ]:
process_ro = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks, args=(df, "ro", 7, "ingforms", CHUNK_SIZE)
)

In [ ]:
process_ro.start()

2026-03-03 14:20:16 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-03 14:20:16 INFO: Downloaded file to /home/christopher/stanza_resources/resources.json
2026-03-03 14:20:18 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-03-03 14:20:18 INFO: Using device: cpu
2026-03-03 14:20:18 INFO: Loading: tokenize
2026-03-03 14:20:20 INFO: Loading: mwt
2026-03-03 14:20:20 INFO: Loading: pos

Parsing chunk 0 to 2499
Parsing chunk 2500 to 4999
Parsing chunk 5000 to 7499
Parsing chunk 7500 to 9999
Parsing chunk 2500 to 4999
Parsing chunk 10000 to 12499
Parsing chunk 12500 to 14999
Parsing chunk 15000 to 17499
Parsing chunk 17500 to 19999
Parsing chunk 5000 to 7499
Parsing chunk 20000 to 22499
Parsing chunk 22500 to 24999
Parsing chunk 25000 to 27499
Parsing chunk 27500 to 29414
Parsing chunk 7500 to 9999
Parsing chunk 10000 to 12499
Parsing chunk 12500 to 14999
Parsing chunk 15000 to 17499
Parsing chunk 17500 to 19999
Parsing chunk 20000 to 22499


In [17]:
parsed_dict_ro = functions_stanza.load_pickled_data(folder="ingforms", language="ro")

parsed_slice_en_27500_29414.pkl
parsed_slice_en_17500_19999.pkl
parsed_slice_ro_20000_22499.pkl
parsed_slice_en_7500_9999.pkl
parsed_slice_ro_17500_19999.pkl
parsed_slice_ro_10000_12499.pkl
parsed_slice_en_20000_22499.pkl
parsed_slice_ro_0_2499.pkl
parsed_slice_ro_15000_17499.pkl
parsed_slice_en_12500_14999.pkl
parsed_slice_en_2500_4999.pkl
parsed_slice_ro_5000_7499.pkl
parsed_slice_ro_25000_27499.pkl
parsed_slice_en_10000_12499.pkl
parsed_slice_en_25000_27499.pkl
parsed_slice_ro_22500_24999.pkl
parsed_slice_ro_12500_14999.pkl
parsed_slice_en_0_2499.pkl
parsed_slice_ro_27500_29414.pkl
parsed_slice_ro_7500_9999.pkl
parsed_slice_en_22500_24999.pkl
parsed_slice_ro_2500_4999.pkl
parsed_slice_en_15000_17499.pkl
parsed_slice_en_5000_7499.pkl


In [18]:
gerunziu_indices_and_words = []

In [19]:
for i in range(0, len(df)):
    gerunzius = functions_stanza.get_gerunzius_in_sentences(parsed_dict_ro[i])
    if gerunzius:
        gerunziu_indices_and_words.append((i, gerunzius))

In [20]:
gerunziu_indices_and_words

[(8, [('Văzând', 'vedea')]),
 (10, [('alunecând', 'aluneca')]),
 (11, [('alunecând', 'aluneca')]),
 (17, [('fiind', 'fi')]),
 (24, [('plimbându', 'plimba'), ('întrebând', 'întreba')]),
 (25, [('plimbându', 'plimba'), ('întrebând', 'întreba')]),
 (26, [('plimbându', 'plimba'), ('întrebând', 'întreba')]),
 (27, [('alergând', 'alerga')]),
 (28, [('bombănind', 'bombăni')]),
 (30, [('încercând', 'încerca'), ('întrebându', 'întrebându')]),
 (31, [('încercând', 'încerca'), ('întrebându', 'întrebându')]),
 (32, [('trăgând', 'trage')]),
 (33, [('trăgând', 'trage')]),
 (34, [('trăgând', 'trage')]),
 (36, [('găsind', 'găsi')]),
 (50, [('întrebându', 'întrebându'), ('ţinându', 'ţina')]),
 (51, [('întrebându', 'întrebându'), ('ţinându', 'ţina')]),
 (52, [('întrebându', 'întrebându'), ('ţinându', 'ţina')]),
 (63, [('ţinând', 'ţinca'), ('bombănind', 'bombăni')]),
 (64, [('ţinând', 'ţinca'), ('bombănind', 'bombăni')]),
 (65, [('ţinând', 'ţinca'), ('bombănind', 'bombăni')]),
 (67, [('spunându', 'spune'

# Building combined df

In [21]:
adv_part_indices_and_lemmas_df = pd.DataFrame(
    adv_part_indices_and_lemmas, columns=["index", "lemma"]
).set_index("index")

In [22]:
gerunziu_indices_and_words_df = pd.DataFrame(
    gerunziu_indices_and_words, columns=["index", "gerunzius_and_lemmas"]
).set_index("index")

In [23]:
gerunziu_indices_and_words_df

,gerunzius_and_lemmas
index,
8,"[(Văzând, vedea)]"
10,"[(alunecând, aluneca)]"
11,"[(alunecând, aluneca)]"
17,"[(fiind, fi)]"
24,"[(plimbându, plimba), (întrebând, întreba)]"
...,...
29409,"[(plutind, pluti)]"
29410,"[(plutind, pluti)]"
29411,"[(plutind, pluti)]"


In [24]:
adv_part_indices_and_lemmas_df

,lemma
index,
27,hurry
31,wonder
36,find
41,find
56,send
...,...
29392,make
29396,splash
29400,cause


In [25]:
df = pd.concat(
    [df, adv_part_indices_and_lemmas_df, gerunziu_indices_and_words_df],
    axis=1,
)

In [26]:
df

,1,2,3,4,5,6,7,8,english,lemma,gerunzius_and_lemmas
0,carroll-alenka_v_kraji,Alice was,beginning,to get very tired of sitting by her sister on ...,carroll-alenka_v_kraji,,"Aşezată lângă sora ei, pe malul apei, Alice în...",,Alice was beginning to get very tired of sitti...,NaN,NaN
1,carroll-alenka_v_kraji,Alice was beginning to get very tired of,sitting,"by her sister on the bank , and of having noth...",carroll-alenka_v_kraji,,"Aşezată lângă sora ei, pe malul apei, Alice în...",,Alice was beginning to get very tired of sitti...,NaN,NaN
2,carroll-alenka_v_kraji,Alice was beginning to get very tired of sitti...,having,nothing to do : once or twice she had peeped i...,carroll-alenka_v_kraji,,"Aşezată lângă sora ei, pe malul apei, Alice în...",,Alice was beginning to get very tired of sitti...,NaN,NaN
3,carroll-alenka_v_kraji,Alice was beginning to get very tired of sitti...,reading,", but it had no pictures or conversations in i...",carroll-alenka_v_kraji,,"Aşezată lângă sora ei, pe malul apei, Alice în...",,Alice was beginning to get very tired of sitti...,NaN,NaN
4,carroll-alenka_v_kraji,So she was,considering,", in her own mind ( as well as she could , for...",carroll-alenka_v_kraji,,Şi doar ce era pe punctul de a chibzui (atât c...,,"So she was considering, in her own mind (as we...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
29410,Rushdie-MauruvPosl_Vzd,to the defeated love that is greater than what...,flowing,"together , for putting an end to frontiers , f...",Rushdie-MauruvPosl_Vzd,,"Alhambra, fortul roşu al Europei, fratele celo...",,to the defeated love that is greater than what...,NaN,"[(plutind, pluti)]"
29411,Rushdie-MauruvPosl_Vzd,to the defeated love that is greater than what...,putting,"an end to frontiers , for the dropping of the ...",Rushdie-MauruvPosl_Vzd,,"Alhambra, fortul roşu al Europei, fratele celo...",,to the defeated love that is greater than what...,NaN,"[(plutind, pluti)]"
29412,Rushdie-MauruvPosl_Vzd,The world is full of sleepers,waiting,for their moment of return :,Rushdie-MauruvPosl_Vzd,,Lumea e plină de cei adormiţi care îşi aşteapt...,,The world is full of sleepers waiting for thei...,NaN,NaN
29413,Rushdie-MauruvPosl_Vzd,"and then , like a latter - day Van Winkle , I'...",according,to our family's old practice of falling asleep...,Rushdie-MauruvPosl_Vzd,,"şi apoi, precum un Van Winkle ajuns la bătrîne...",,"and then, like a latter-day Van Winkle, I'll l...",NaN,"[(sperînd, sperî)]"


In [27]:
df["is_adv_participle"] = False
df.loc[~df["lemma"].isna(), "is_adv_participle"] = True

In [28]:
df["has_gerunziu"] = False
df.loc[~df["gerunzius_and_lemmas"].isna(), "has_gerunziu"] = True

In [29]:
df.loc[df["is_adv_participle"] & df["has_gerunziu"]]

,1,2,3,4,5,6,7,8,english,lemma,gerunzius_and_lemmas,is_adv_participle,has_gerunziu
27,carroll-alenka_v_kraji,"before her was another long passage , and the ...",hurrying,down it .,carroll-alenka_v_kraji,,"înaintea ei se întindea un coridor lung, la ca...",,"before her was another long passage, and the W...",hurry,"[(alergând, alerga)]",True,True
31,carroll-alenka_v_kraji,and when Alice had been all the way down one s...,wondering,how she was ever to get out again .,carroll-alenka_v_kraji,,"Şi, după ce-o străbătu în lung şi în lat, înce...",,and when Alice had been all the way down one s...,wonder,"[(încercând, încerca), (întrebându, întrebându)]",True,True
36,carroll-alenka_v_kraji,"However , this bottle was not marked ‘ poison ...",finding,"it very nice , ( it had , in fact , a sort of ...",carroll-alenka_v_kraji,,"Totuşi pe această sticlă nu scria "" Otravă "" ş...",,"However, this bottle was not marked ‘poison,’ ...",find,"[(găsind, găsi)]",True,True
65,carroll-alenka_v_kraji,"It was the White Rabbit returning , splendidly...",muttering,"to himself as he came , ‘ Oh ! the Duchess , t...",carroll-alenka_v_kraji,,"Era, şi de data asta, Iepurele Alb. Nespus de ...",,"It was the White Rabbit returning, splendidly ...",mutter,"[(ţinând, ţinca), (bombănind, bombăni)]",True,True
68,carroll-alenka_v_kraji,"Alice took up the fan and gloves and , as the ...",talking,.,carroll-alenka_v_kraji,,Alice adună de pe jos mănuşile şi evantaiul şi...,,"Alice took up the fan and gloves and, as the h...",talk,"[(spunându, spune)]",True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29389,Rushdie-MauruvPosl_Vzd,When I had brought my story to the X - ray roo...,humming,a talcum - powder shanty .,Rushdie-MauruvPosl_Vzd,,După ce ajunsei cu povestirea la încăperea pli...,,When I had brought my story to the X-ray room ...,hum,"[(zdrăngănindu, zdrăngăni), (ţinînd, ţine), (f...",True,True
29392,Rushdie-MauruvPosl_Vzd,"My breath brayed loudly ,",making,a jackass of me once again .,Rushdie-MauruvPosl_Vzd,,"Respiraţia îmi era ca un nechezat puternic, fă...",,"My breath brayed loudly, making a jackass of m...",make,"[(făcîndu, face)]",True,True
29396,Rushdie-MauruvPosl_Vzd,– Bloodstains spread across the front and rear...,splashing,"in his own , and fatal , pools .",Rushdie-MauruvPosl_Vzd,,-Petele de sînge se întinseră pe partea din fa...,,– Bloodstains spread across the front and rear...,splash,"[(împroşcînd, împroşca)]",True,True
29405,Rushdie-MauruvPosl_Vzd,"I put on my greatcoat , and ,",leaving,"my cell , found the rest of my text in Vasco's...",Rushdie-MauruvPosl_Vzd,,"Îmi îmbrăcai haina şi, ieşind din celulă, găsi...",,"I put on my greatcoat, and, leaving my cell, f...",leave,"[(ieşind, ieşi)]",True,True


In [30]:
df.loc[df["is_adv_participle"] & df["has_gerunziu"]].to_csv("extraction_output/adv_part_with_gerunziu.csv")

In [31]:
len(df)

29415

In [32]:
len(df.loc[df["is_adv_participle"]])

5934

In [33]:
df.to_csv("extraction_output/annotated_all_part_to_ger.csv")

# Assigning adverbial participles to gerunzius

In [35]:
df = pd.read_csv("extraction_output/adv_part_with_gerunziu.csv", index_col=0)

In [36]:
import functions_gensim

In [37]:
for (
    i,
    row,
) in df.iterrows():
    adv_part = row["3"]
    lemma_en = row["lemma"]
    gerunzius_and_lemmas = eval(row["gerunzius_and_lemmas"])
    best_match = functions_gensim.get_best_match(
        word=adv_part,
        lemma=lemma_en,
        candidate_words_and_lemmas=gerunzius_and_lemmas,
        language_word="en",
    )
    df.loc[i, ["best_match", "best_score"]] = best_match[0], best_match[1]

In [38]:
df.to_csv("extraction_output/adv_ger_matches.csv")